In [1]:
import os
import sys
import copy

import ifcopenshell
import numpy as np
import multiprocessing
import math
import pickle

from tqdm import tqdm
from ifcopenshell import geom
from pxr import Gf, Kind, Sdf, Usd, UsdGeom, UsdShade


ModuleNotFoundError: No module named 'ifcopenshell'

In [2]:
# IFCはZ-UPとなる
Y_UP = False

# ここを任意のパスに変えてください
ifc_path = 'files/ToyodaLab.ifc'

# IfcOpenShellの設定
try:
    ifc_file = ifcopenshell.open(ifc_path)
except:
    print(ifcopenshell.get_log())
else:
    # https://blenderbim.org/docs-python/ifcopenshell/geometry_settings.html
    settings = geom.settings()
    # 世界座標系に変換
    # settings.set(settings.USE_WORLD_COORDS,True)
    # これがないとnormalsが破棄される
    settings.set(settings.WELD_VERTICES, False)
    settings.set(settings.APPLY_DEFAULT_MATERIALS, True)    

In [3]:
def format_ifc_info(info) -> dict:
    """ IfcOpenShellの情報はmultiprocessingを含んでいるので一般オブジェクトに変換する
    
    Args:
        info (_type_): IfcEntityのメタデータ

    Returns:
        dict: _description_
    """
    ret = dict()
    for key in info.keys():
        item = info.get(key)
        if type(item) != ifcopenshell.entity_instance and item != None:
            ret[key] = item
    return ret


def get_project_info(ifc_file, settings, name='Sample') :
    """ プロジェクトの情報を取得する

    Args:
        ifc_file (_type_): _description_
        settings (_type_): _description_
        name (str, optional): _description_. Defaults to 'Sample'.

    Returns:
        str, str, str: _description_
    """
    import math
    # IfcProjectから名前をとる
    prj = ifc_file.by_type('IfcProject')[0]
    name_ = prj.LongName if prj.LongName != 'プロジェクト名' else name
    name_ = name if name_ is None else name_
    
    # IfcSiteから緯度経度を取得
    site = ifc_file.by_type('IfcSite')[0]
    lat = '.'.join([str(i) for i in site.RefLatitude]) if site.RefLatitude is not None else ""
    lon = '.'.join([str(i) for i in site.RefLongitude]) if site.RefLongitude is not None else ""
 
    return name_, lat, lon


def get_geometry(settings, ifc_file, materials):
    """対象のオブジェクトのジオメトリを取得。ジェネレータに対応。

    Args:
        settings (_type_): ifcopenshellの設定
        ifc_file (_type_): 対象のIFCオブジェクト

    Yields:
        _type_: _description_
    """

    iterator = geom.iterator(settings, ifc_file, multiprocessing.cpu_count())

    grouped_verts = []
    grouped_norms = []

    if iterator.initialize():
        while True:
            shape = iterator.get()
            element = ifc_file.by_guid(shape.guid)
            
            # 開口部エレメントはジオメトリからは除く
            # TODO IfcSpatialZone
            if element.is_a("IfcOpeningElement") or element.is_a("IfcSpace") or element.is_a("IfcSpatialZone"): 
                if not iterator.next():
                    break
                continue
            info = format_ifc_info(element.get_info())
            
            # 変換マトリックス
            matrix = shape.transformation.matrix.data
            # メッシュの取得
            verts = shape.geometry.verts
            # indices
            indices = shape.geometry.faces

            # これは、頂点法線になっている。
            # 全て反転している気がする。バグ？？
            norms = [n*-1 for n in shape.geometry.normals]

            _materials = shape.geometry.materials # Material names and colour style information that are relevant to this shape
            
            
            if not Y_UP:
                grouped_verts = [(verts[i], verts[i+1], verts[i+2]) for i in range(0, len(verts), 3)]
                grouped_norms = [(norms[i], norms[i+1], norms[i+2]) for i in range(0, len(norms), 3)]
            else:
                grouped_verts = [(verts[i], verts[i+2], verts[i+1]) for i in range(0, len(verts), 3)]
                grouped_norms = [(norms[i], norms[i+2], norms[i+1]) for i in range(0, len(norms), 3)]
 

            # USDのfacevaringに対応させるため、並べ替えを行う。
            grouped_norms = [grouped_norms[f] for f in indices]
            
            diffuse_color = (0,0,0)
            if len(_materials) != 0:
                # マテリアルは複数ないと仮定する
                for mat in _materials:
                    # 日本語が文字化けしている気がする
                    material_name = mat.name.replace('-', '_').replace('<', '').replace('>', '').replace('/', '').replace(',','').replace('(','').replace(')','')
                    if material_name in materials:
                        continue
                    diffuse_color = mat.diffuse
                    specular_color = mat.specular if mat.has_specular else None
                    transparency = mat.transparency if mat.has_transparency else None
                    # IfcWindowくらいは透過させようと思った
                    if element.is_a("IfcWindow") :
                        transparency = 0.8
                    materials[material_name] = (diffuse_color, specular_color, transparency)
                    
            else:
                #color = [0,0,0]
                material_name = None
            yield  grouped_verts, indices, grouped_norms, info, material_name, diffuse_color, matrix
            if not iterator.next():
                break



def get_properties(element):
    """ IFCのオブジェクトよりプロパティを抽出

    Args:
        element (_type_): _description_

    Returns:
        dict : プロパティセット
    """
    ret = vars(element)

    # propertyがない場合はエラーになる
    if hasattr(element, 'IsDefinedBy'):
        for tmp in element.IsDefinedBy:
            if tmp.is_a('IfcRelDefinesByProperties'):            
                pset = tmp.RelatingPropertyDefinition
                if pset.is_a('IfcPropertySet'):
                    for prop in pset.HasProperties:
                        try:
                            name = prop.Name
                            val = prop.NominalValue
                            ret[name] = val.wrappedValue
                        except:
                            print('Invalid Property: ' + name)
                elif pset.is_a('IfcElementQuantity'):
                    # 面積の場合がある
                    name = pset.Name
                    measurement = pset.MethodOfMeasurement 
                    quantities = pset.Quantities[0] 
                    #from IPython.core.debugger import Pdb; Pdb().set_trace()
                    if quantities.is_a('IfcQuantityArea'):
                        label = quantities.Name.replace(' ', '_')
                        description = quantities.Description
                        unit = quantities.Unit
                        # 記法の変更
                        area_value = quantities.AreaValue
                        ret[label]  = area_value
                else:
                    print('Invalid Type: ' + vars(pset)['type'])
            elif tmp.is_a('IfcRelDefinesByType'):
                #TODO クラス（Family）の定義
                pass
            else:
                print('Invalid Type : ' + vars(tmp)['type'])
      
    # 不要なプロパティの刈込(主観で決めている)
    ret.pop('OwnerHistory', None)
    ret.pop('CompositionType', None)
    ret.pop('Representation', None)
    ret.pop('ObjectPlacement', None)
    # ObjectTypeとほぼ等価
    ret.pop('Reference', None)
    # その他
    addr = ret.pop('BuildingAddress', None)
    if addr:
        addr = addr.AddressLines[0]
        ret['Address'] = addr
    return ret


In [4]:
shapes = dict()
materials = {}
geometries = {}

for (verts, indices, norms, info, material, color, translate) in tqdm(get_geometry(settings, ifc_file, materials)):
    guid = info['GlobalId']
    faces = [3 for _ in range(int(len(indices)/3))]
    geometries[guid] = [faces, verts, indices, material, color, norms, translate]

# 一時保存
with open('output/{0}_tmp.pickle'.format(ifc_path.split('/')[1].split('.')[0]), 'wb') as f:
    pickle.dump((geometries, materials), f)

76it [00:02, 27.30it/s] 


In [5]:
def create_mesh(stage, path, geometry, materialPrims, offset=[0,0,0]):
    faces = geometry[0]
    vertices = geometry[1]
    indices = geometry[2]
    mat = materialPrims[geometry[3]]
    color = geometry[4]
    normals = geometry[5]
    
    mesh = UsdGeom.Mesh.Define(stage, path + "/mesh")
    mesh.CreatePointsAttr(vertices)
    mesh.CreateFaceVertexCountsAttr(faces)
    mesh.CreateFaceVertexIndicesAttr(indices)
    
    # BoundingBoxをセット？
    mesh.CreateExtentAttr(UsdGeom.PointBased(mesh).ComputeExtent(mesh.GetPointsAttr().Get()))

    # 法線はデフォルトではCatmull-clarkによって細かくリメッシュされる。
    mesh.CreateNormalsAttr(normals)
    mesh.CreateSubdivisionSchemeAttr("none")
    mesh.SetNormalsInterpolation(UsdGeom.Tokens.faceVarying)

    # DoubleSideはFalse
    mesh.CreateDoubleSidedAttr(False)

    # displayColorはBlenderでは反映して表示しない。
    colorAttr = mesh.GetDisplayColorAttr()
    colorAttr.Set([Gf.Vec3f(color[0], color[1], color[2])])  
    # これがないとusdviewでエラーが出る  
    mesh.GetPrim().ApplyAPI(UsdShade.MaterialBindingAPI)
    UsdShade.MaterialBindingAPI(mesh).Bind(mat, UsdShade.Tokens.preview)
    

def set_custom_data(stage, prim,props):
    target = stage.GetPrimAtPath(prim.GetPath())
    target.SetCustomDataByKey('class', props['type'])
    target.SetCustomDataByKey('GUID', props['GlobalId'])

    if 'Name' in props:
        target.SetCustomDataByKey('Name', props['Name']) 
    if 'LongName' in props:
        target.SetCustomDataByKey('LongName', props['LongName']) 
    if 'Description' in props:
        target.SetCustomDataByKey('Description', props['Description'])
    # Siteに対して
    if 'RefLatitude' in props:
        lat = '.'.join([str(i) for i in props['RefLatitude']]) if props['RefLatitude'] is not None else ""
        lon = '.'.join([str(i) for i in props['RefLongitude']]) if props['RefLongitude'] is not None else ""
        target.SetCustomDataByKey('Latitude', lat)
        target.SetCustomDataByKey('Longitude', lon)
    if 'Properties' in props:
        # TODO　シングルプロパティとそうでないものがある
        None
        

In [6]:
with open('output/{0}_tmp.pickle'.format(ifc_path.split('/')[1].split('.')[0]), 'rb') as f:
    t = pickle.load(f)
    geometries = t[0]
    materials = t[1]

In [7]:
output_usd_path = 'output/{0}_structured.usda'.format(ifc_path.split('/')[1].split('.')[0])

if os.path.isfile(output_usd_path):
    os.remove(output_usd_path)

stage = None
stage = Usd.Stage.CreateInMemory()
#stage = Usd.Stage.CreateNew(output_usd_path)
# シーンの単位。以下でm単位となる。
# https://ft-lab.jp/blog_3dcg/?p=2005
UsdGeom.SetStageMetersPerUnit(stage, 1.0)

if Y_UP:
    UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.y)
else:
    UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.z)

model_root = stage.DefinePrim("/IFC_Model", "Xform")
Usd.ModelAPI(model_root).SetKind(Kind.Tokens.assembly)
stage.SetDefaultPrim(model_root)
references = dict()

materialPrims = {}

# 先にマテリアルを作成
# TODO 
metallic = 0.0
roughness = 1.0

for name in materials:
    path = Sdf.Path(f"/Materials/{name}")
    mat = UsdShade.Material.Define(stage, path)
    #shader = UsdShade.Material(mat)
    shader = UsdShade.Shader.Define(stage, path.AppendChild('PBRShader'))
    shader.CreateIdAttr().Set("UsdPreviewSurface")
    shader.CreateIdAttr('UsdPreviewSurface')
    color = materials[name][0]
    shader.CreateInput('diffuseColor', Sdf.ValueTypeNames.Color3f).Set(Gf.Vec3f(color[0],color[1],color[2]))
    shader.CreateInput("roughness", Sdf.ValueTypeNames.Float).Set(roughness)
    shader.CreateInput("metallic", Sdf.ValueTypeNames.Float).Set(metallic)
    if materials[name][1]:
        color = materials[name][1]
        shader.CreateInput('specularColor', Sdf.ValueTypeNames.Color3f).Set(Gf.Vec3f(color[0],color[1],color[2]))
    if materials[name][2]:
        # IFCではtransparencyなので変換する。
        shader.CreateInput('opacity', Sdf.ValueTypeNames.Float).Set(1.0 - materials[name][2]) 
        shader.CreateInput("roughness", Sdf.ValueTypeNames.Float).Set(0.0)
    mat.CreateSurfaceOutput().ConnectToSource(shader.ConnectableAPI(), "surface")
    materialPrims[name] = mat


In [8]:

def append_prim(stage, props, path, geometries, kind=Kind.Tokens.group):
    prim = UsdGeom.Xform.Define(stage, path)
    Usd.ModelAPI(prim).SetKind(Kind.Tokens.group)
    set_custom_data(stage, prim,props)

    if props['GlobalId'] in geometries:
        # 座標変換
        geom = copy.copy(geometries[props['GlobalId']])
        verts = geom[1]
        t = geom[6]
        tmp = [(t[i],t[i+1],t[i+2]) for i in range(0, 12, 3)]
        offset = tmp[3]
        M = np.matrix((tmp[0], tmp[1], tmp[2])).T
        geom[1] = [np.ravel(np.dot(M,np.array(vert))).tolist() for vert in verts] 
        UsdGeom.XformCommonAPI(prim).SetTranslate(offset)
        Usd.ModelAPI(prim).SetKind(Kind.Tokens.component)
        stage.GetPrimAtPath(prim.GetPath()).SetCustomDataByKey('GUID', props['GlobalId'])
        create_mesh(stage, str(prim.GetPath()), geom, materialPrims)
    return prim
    

props = get_properties(ifc_file.by_type('IfcSite')[0])
site = append_prim(stage, props, str(model_root.GetPath())+'/Site',geometries)

props = get_properties(ifc_file.by_type('IfcBuilding')[0])
building = append_prim(stage, props, str(site.GetPath())+'/Building', geometries)

storeys = ifc_file.by_type('IfcBuildingStorey')

def proc_elements(model, prim):
    #element_num = 0
    for elem in model.ContainsElements:
        for e in elem.RelatedElements:
            props = get_properties(e)
            id = props['id']
            elem_prim = append_prim(stage, props, str(prim.GetPath())+f'/Element_{id}', geometries)
            #element_num += 1
            if len(e.IsDecomposedBy) > 0:
                objects = e.IsDecomposedBy[0].RelatedObjects
                # object_num = 0
                for obj_model in objects:
                    props = get_properties(obj_model)
                    id = props['id']
                    obj_prim = append_prim(stage, props, str(elem_prim.GetPath())+f'/Object_{id}', geometries)
                    #object_num += 1
                    
#storey_num = 0
for storey_model in tqdm(storeys):
    # IfcBuildingStorey
    props = get_properties(storey_model)
    id = props['id']
    storey_prim = append_prim(stage, props, str(building.GetPath())+f'/Storey_{id}', geometries)
    #storey_num += 1
    
    if storey_model.ContainsElements:
        proc_elements(storey_model, storey_prim)
    
    if len(storey_model.IsDecomposedBy) > 0:
        #space_num = 0
        assert(len(storey_model.IsDecomposedBy) == 1)
        spaces = storey_model.IsDecomposedBy[0].RelatedObjects
        for space_model in spaces:
            props = get_properties(space_model)
            id = props['id']
            space = append_prim(stage, props, str(storey_prim.GetPath())+f'/Space_{id}', geometries)
            #space_num += 1            
            if space_model.ContainsElements:
                proc_elements(space_model, space)
                        
# stage.ExportToString()
stage.Export(output_usd_path)

100%|██████████| 2/2 [00:01<00:00,  1.15it/s]


True

In [9]:

stage.Save()